In [ ]:
# [Cell 1] 환경 설정 및 폴더 구조 생성
from google.colab import drive
import os

# 1. Google Drive 마운트
drive.mount('/content/drive')

# 2. 프로젝트 경로 설정
PROJECT_PATH = '/content/drive/MyDrive/super_solutioner/LNO_base'
DATA_PATH = os.path.join(PROJECT_PATH, 'data/DIV2K')
CKPT_PATH = os.path.join(PROJECT_PATH, 'checkpoints')
RESULT_PATH = os.path.join(PROJECT_PATH, 'results')
""
# 3. 폴더 생성
os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(CKPT_PATH, exist_ok=True)
os.makedirs(RESULT_PATH, exist_ok=True)

print(f"Project initialized at: {PROJECT_PATH}")

In [ ]:
# [Cell 2] DIV2K 데이터셋 다운로드 (이미 있다면 스킵됨)
import urllib.request
import zipfile

def download_and_extract(url, dest_path):
    filename = url.split('/')[-1]
    file_path = os.path.join(dest_path, filename)

    if not os.path.exists(file_path):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, file_path)
        print("Download complete. Extracting...")
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            zip_ref.extractall(dest_path)
        print("Extraction complete.")
    else:
        print(f"{filename} already exists. Skipping.")

# DIV2K Train HR 다운로드 (약 3.5GB, 시간이 좀 걸릴 수 있음)
# 실습을 빠르게 하려면 적은 용량의 데이터셋으로 대체 가능하나, 여기선 정석대로 진행
url_train_hr = "http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip"
url_valid_hr = "http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip"

download_and_extract(url_train_hr, DATA_PATH)
download_and_extract(url_valid_hr, DATA_PATH)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import glob
import os
import random

class DIV2KDataset(Dataset):
    def __init__(self, root_dir, phase='train', patch_size=48, max_scale=4):
        self.phase = phase
        self.max_scale = max_scale
        # HR 패치 크기는 가장 큰 배율(x4)을 기준으로 넉넉하게 잡습니다.
        # 예: LR 48 -> HR 192 (x4 기준)
        self.hr_patch_size = patch_size * max_scale

        subset = 'DIV2K_train_HR' if phase == 'train' else 'DIV2K_valid_HR'
        # 파일 경로 리스트 생성
        self.image_paths = sorted(glob.glob(os.path.join(root_dir, subset, '*.png')))
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # [수정 1] 여기서 이미지를 먼저 로드해야 합니다! (이 줄이 없어서 에러 발생)
        hr_img = Image.open(self.image_paths[idx]).convert('RGB')

        # Train: Random Crop (HR 기준 크기로 자름)
        if self.phase == 'train':
            w, h = hr_img.size

            # 이미지가 패치보다 작으면 리사이즈
            if w < self.hr_patch_size or h < self.hr_patch_size:
                 hr_img = hr_img.resize((max(w, self.hr_patch_size), max(h, self.hr_patch_size)), Image.BICUBIC)
                 w, h = hr_img.size

            x1 = random.randint(0, w - self.hr_patch_size)
            y1 = random.randint(0, h - self.hr_patch_size)
            hr_patch = hr_img.crop((x1, y1, x1 + self.hr_patch_size, y1 + self.hr_patch_size))

            # Augmentation
            if random.random() < 0.5: hr_patch = hr_patch.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() < 0.5: hr_patch = hr_patch.transpose(Image.FLIP_TOP_BOTTOM)

            return self.to_tensor(hr_patch)

        else:
            # Valid: 테스트용이니 너무 크면 메모리 터질 수 있어 적당히 자르거나 그대로 사용
            # 여기서는 편의상 1024x1024 센터 크롭 혹은 리사이즈 안 함
            return self.to_tensor(hr_img)

# [수정 2] 데이터 로더 테스트 코드 수정
# (Multi-scale 방식에서는 Dataset이 HR만 반환하므로 lr, hr 2개를 받으면 에러남)
train_dataset = DIV2KDataset(DATA_PATH, phase='train', patch_size=48, max_scale=4)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)

print(f"Train Dataset Size: {len(train_dataset)}")

# HR 이미지 하나만 반환되므로 변수 하나로 받음
sample_hr = train_dataset[0]
print(f"Sample HR shape: {sample_hr.shape}") # [3, 192, 192] 예상

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# 1. 기본 LNO Layer (기존 PR2d_SR 유지 또는 동일)
class PR2d_SR(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super(PR2d_SR, self).__init__()
        self.modes1 = modes1
        self.modes2 = modes2
        self.scale = (1 / (in_channels * out_channels))

        # Weights
        self.weights_pole1 = nn.Parameter(self.scale * torch.randn(in_channels, out_channels, self.modes1, dtype=torch.cfloat))
        self.weights_pole2 = nn.Parameter(self.scale * torch.randn(in_channels, out_channels, self.modes2, dtype=torch.cfloat))
        self.weights_residue = nn.Parameter(self.scale * torch.randn(in_channels, out_channels, self.modes1, self.modes2, dtype=torch.cfloat))

    def output_PR(self, lambda1, lambda2, alpha, weights_pole1, weights_pole2, weights_residue):
        term1 = lambda1 - weights_pole1
        term2 = lambda2 - weights_pole2
        denom = term1.unsqueeze(-1) * term2.unsqueeze(-2)

        # [수정] 분모가 0이 되는 것을 방지하기 위해 epsilon 추가 (안전장치)
        epsilon = 1e-5
        # 복소수 나눗셈 안정을 위해 크기가 너무 작으면 epsilon 더함 (간단한 방식)
        denom = torch.where(torch.abs(denom) < epsilon, denom + epsilon, denom)

        H = torch.div(weights_residue, denom)
        out_residue = torch.einsum("bixy,ioxy->boxy", alpha, H)
        return out_residue

    def forward(self, x, target_size):
        B, C, H, W = x.shape
        alpha = torch.fft.fft2(x, dim=[-2, -1])
        alpha_modes = alpha[..., :self.modes1, :self.modes2]

        omega1 = torch.fft.fftfreq(self.modes1, d=1/self.modes1).to(x.device) * 2 * np.pi * 1j
        omega2 = torch.fft.fftfreq(self.modes2, d=1/self.modes2).to(x.device) * 2 * np.pi * 1j
        lambda1 = omega1.reshape(1, 1, self.modes1)
        lambda2 = omega2.reshape(1, 1, self.modes2)

        out_res = self.output_PR(lambda1, lambda2, alpha_modes,
                                 self.weights_pole1, self.weights_pole2, self.weights_residue)

        x_out = torch.fft.ifft2(out_res, s=target_size, dim=[-2, -1])
        return torch.real(x_out)

# 2. [신규] Residual LNO Block (깊게 쌓기 위한 블록)
class LNOBlock(nn.Module):
    def __init__(self, width, modes1, modes2):
        super().__init__()
        self.norm1 = nn.InstanceNorm2d(width)
        # 이 블록 안에서는 해상도 변경 없이 특징 추출만 수행 (Target=Input size)
        self.lno = PR2d_SR(width, width, modes1, modes2)
        self.act = nn.GELU()

        self.norm2 = nn.InstanceNorm2d(width)
        self.mlp = nn.Sequential(
            nn.Conv2d(width, width * 2, 1),
            nn.GELU(),
            nn.Conv2d(width * 2, width, 1)
        )

    def forward(self, x):
        # LNO Path (Resolution Preserving here)
        # Block 내부에서는 입력 크기 그대로 유지하며 특징만 학습
        target_size = (x.shape[2], x.shape[3])
        resid = self.lno(self.norm1(x), target_size)
        x = x + self.act(resid)

        # MLP Path
        resid = self.mlp(self.norm2(x))
        x = x + resid
        return x

# 3. [업그레이드] Deep LNO Model
class DeepLNO_SR(nn.Module):
    def __init__(self, in_channels=3, width=64, layers=4, modes1=32, modes2=32):
        super(DeepLNO_SR, self).__init__()

        # 1. Lifting
        self.lifting = nn.Conv2d(in_channels, width, 1)

        # 2. Deep Feature Extraction (여러 층 쌓기)
        self.layers = nn.ModuleList([
            LNOBlock(width, modes1, modes2) for _ in range(layers)
        ])

        # 3. Upsampling Layer (마지막에 한 번 LNO로 업샘플링)
        self.upsampler = PR2d_SR(width, width, modes1, modes2)

        # 4. Projection
        self.proj = nn.Conv2d(width, in_channels, 1)

    def forward(self, x, target_size=None):
        if target_size is None:
            target_size = (x.shape[2]*4, x.shape[3]*4)

        # Lift
        x_feat = self.lifting(x)

        # Deep Features (LR 해상도에서 특징 정제)
        for layer in self.layers:
            x_feat = layer(x_feat)

        # Upsample via LNO (여기서 해상도 변경)
        x_up = self.upsampler(x_feat, target_size=target_size)

        # Project to RGB
        out = self.proj(x_up)

        # Global Residual (Bicubic)
        base = F.interpolate(x, size=target_size, mode='bicubic', align_corners=False)

        return out + base

In [ ]:
# [Cell 5] Optimizer 및 Loss 정의
import math
from typing import List
from torch import Tensor
from torch.optim.optimizer import Optimizer

# --- Provided Adam Implementation ---
def adam(params, grads, exp_avgs, exp_avg_sqs, max_exp_avg_sqs, state_steps, *,
         amsgrad, beta1, beta2, lr, weight_decay, eps):
    # (Implementation copied from provided Adam.py content for self-containment)
    for i, param in enumerate(params):
        grad = grads[i]
        exp_avg = exp_avgs[i]
        exp_avg_sq = exp_avg_sqs[i]
        step = state_steps[i]

        bias_correction1 = 1 - beta1 ** step
        bias_correction2 = 1 - beta2 ** step

        if weight_decay != 0:
            grad = grad.add(param, alpha=weight_decay)

        exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
        exp_avg_sq.mul_(beta2).addcmul_(grad, grad.conj(), value=1 - beta2)
        if amsgrad:
            torch.maximum(max_exp_avg_sqs[i], exp_avg_sq, out=max_exp_avg_sqs[i])
            denom = (max_exp_avg_sqs[i].sqrt() / math.sqrt(bias_correction2)).add_(eps)
        else:
            denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(eps)
        step_size = lr / bias_correction1
        param.addcdiv_(exp_avg, denom, value=-step_size)

# --- Spectral Loss Definition ---
class SpectralLoss(nn.Module):
    def __init__(self):
        super(SpectralLoss, self).__init__()
        self.l1 = nn.L1Loss()

    def forward(self, pred, target):
        # Compute FFT
        pred_fft = torch.fft.fft2(pred, dim=[-2, -1], norm='ortho')
        target_fft = torch.fft.fft2(target, dim=[-2, -1], norm='ortho')

        # Compare Magnitude (Amplitude)
        pred_mag = torch.abs(pred_fft)
        target_mag = torch.abs(target_fft)

        return self.l1(pred_mag, target_mag)

# Loss Combination
criterion_pixel = nn.L1Loss().cuda()
criterion_freq = SpectralLoss().cuda()

In [ ]:
# [Cell 8] 하이퍼파라미터 관리 클래스
class Config:
    # 1. 데이터 관련
    SCALE_FACTOR = 4        # 업스케일링 배수
    PATCH_SIZE = 48         # 학습 시 LR 이미지 패치 크기 (HR은 x4인 192가 됨)
    BATCH_SIZE = 32         # 배치 사이즈 (VRAM에 따라 조절: T4 기준 16~32 적당)
    NUM_WORKERS = 2         # 데이터 로더 워커 수

    # 2. 모델 아키텍처 (LNO)
    WIDTH = 64              # 채널 수 (Embedding Dimension)
    MODES1 = 32             # Fourier/Laplace Mode 수 (H축)
    MODES2 = 32             # Fourier/Laplace Mode 수 (W축)
    LAYERS = 4

    # 3. 학습 관련
    LR = 1e-5               # 초기 학습률
    EPOCHS = 2000            # 총 학습 에폭
    step_size = 50          # Scheduler Step
    gamma = 0.5             # Scheduler Decay

    # 4. 경로 및 기타
    SEED = 42
    PRINT_FREQ = 10         # 몇 배치마다 로그 출력할지
    SAVE_FREQ = 50           # 몇 에폭마다 저장할지
    CKPT_DIR = '/content/drive/MyDrive/super_solutioner/LNO_base/checkpoints'
    # 계속 덮어씌울 메인 체크포인트 파일명
    CKPT_NAME = "epoch_00.pt"

    # Loss 가중치
    LAMBDA_PIXEL = 1.0
    LAMBDA_FREQ = 0.5

print("Configuration Loaded.")

In [ ]:
import os
import time
import random
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

# [중요] 스케줄러 관련 모듈 임포트
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
def train_model(config):
    # 1. 시드 설정
    torch.manual_seed(config.SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(config.SEED)

    # 2. 데이터 로더
    train_dataset = DIV2KDataset(DATA_PATH, phase='train',
                                 patch_size=config.PATCH_SIZE, max_scale=config.SCALE_FACTOR)
    train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE,
                              shuffle=True, num_workers=config.NUM_WORKERS)

    # 3. 모델 초기화
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = DeepLNO_SR(in_channels=3, width=config.WIDTH,
                       modes1=config.MODES1, modes2=config.MODES2, layers=config.LAYERS).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=config.LR, weight_decay=1e-4)
    warmup_epochs = 10
    scheduler_warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_epochs)
    scheduler_cosine = CosineAnnealingLR(optimizer, T_max=config.EPOCHS - warmup_epochs, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_cosine], milestones=[warmup_epochs])

    criterion_pixel = nn.L1Loss().to(device)
    criterion_freq = SpectralLoss().to(device)

    scales = [2, 3, 4]

    # 시각화용 이미지 로드 (생략 없이 유지)
    val_img_path = os.path.join(DATA_PATH, 'DIV2K_valid_HR', '0801.png')
    if not os.path.exists(val_img_path):
        val_imgs = sorted(glob.glob(os.path.join(DATA_PATH, 'DIV2K_valid_HR', '*.png')))
        if val_imgs: val_img_path = val_imgs[0]

    if os.path.exists(val_img_path):
        val_hr_pil = Image.open(val_img_path).convert('RGB')
        w, h = val_hr_pil.size
        crop = 512
        if w > crop and h > crop:
            val_hr_pil = val_hr_pil.crop((0, 0, crop, crop))
        w, h = val_hr_pil.size
        w, h = w - (w%2), h - (h%2)
        val_hr_pil = val_hr_pil.resize((w, h), Image.BICUBIC)
        val_lr_pil = val_hr_pil.resize((w//config.SCALE_FACTOR, h//config.SCALE_FACTOR), Image.BICUBIC)
        val_hr_tensor = transforms.ToTensor()(val_hr_pil).unsqueeze(0).to(device)
        val_lr_tensor = transforms.ToTensor()(val_lr_pil).unsqueeze(0).to(device)
        val_target_size = (h, w)
    else:
        val_hr_tensor = None


    # 4. Resume Logic (완벽함)
    start_epoch = 0
    best_loss = float('inf')

    if not os.path.exists(config.CKPT_DIR):
        os.makedirs(config.CKPT_DIR)

    ckpt_path = os.path.join(config.CKPT_DIR, config.CKPT_NAME)

    if os.path.exists(ckpt_path):
        print(f"🔄 Found checkpoint at {ckpt_path}. Resuming training...")
        checkpoint = torch.load(ckpt_path)

        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            model.load_state_dict(checkpoint)

        if 'epoch' in checkpoint:
            start_epoch = checkpoint['epoch'] + 1
            print(f"▶ Resuming from Epoch {start_epoch}")
        else:
            print("⚠️ Warning: 'epoch' info not found. Starting from 0.")
            start_epoch = 0
    else:
        print(f"🆕 Starting from scratch (File: {config.CKPT_NAME})")

    # 5. Training Loop
    print(f"🚀 Start Training: Total {config.EPOCHS} Epochs")

    for epoch in range(start_epoch, config.EPOCHS):
        model.train()

        epoch_total_loss = 0.0
        epoch_pixel_loss = 0.0
        epoch_freq_loss = 0.0
        epoch_start = time.time()

        for i, hr_imgs in enumerate(train_loader):
            hr_imgs = hr_imgs.to(device)
            scale = random.choice(scales)

            h_lr = hr_imgs.shape[2] // scale
            w_lr = hr_imgs.shape[3] // scale
            lr_imgs = F.interpolate(hr_imgs, size=(h_lr, w_lr), mode='bicubic', align_corners=False)
            target_size = (hr_imgs.shape[2], hr_imgs.shape[3])

            optimizer.zero_grad()
            preds = model(lr_imgs, target_size=target_size)

            loss_pix = criterion_pixel(preds, hr_imgs)
            loss_freq = criterion_freq(preds, hr_imgs)

            total_loss = (config.LAMBDA_PIXEL * loss_pix) + (config.LAMBDA_FREQ * loss_freq)

            total_loss.backward()

            # [1. Gradient Clipping 복구] 매우 중요!
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            epoch_total_loss += total_loss.item()
            epoch_pixel_loss += loss_pix.item()
            epoch_freq_loss += loss_freq.item()

        scheduler.step()

        # Logging
        avg_total = epoch_total_loss / len(train_loader)
        avg_pix = epoch_pixel_loss / len(train_loader)
        avg_freq = epoch_freq_loss / len(train_loader)

        epoch_time = time.time() - epoch_start
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch [{epoch+1}/{config.EPOCHS}] "
              f"Time: {epoch_time:.1f}s | "
              f"LR: {current_lr:.2e} | "
              f"Total: {avg_total:.5f} (Pix: {avg_pix:.5f}, Freq: {avg_freq:.5f})")

        # 6. Save Checkpoint & Visualization
        if (epoch + 1) % config.SAVE_FREQ == 0:
            checkpoint_dict = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_loss': avg_total
            }
            # 메인 파일 저장 (덮어쓰기)
            torch.save(checkpoint_dict, ckpt_path)

            # [2. 백업 파일 저장 추가] 안전장치!
            backup_filename = f"epoch_{epoch+1}.pth"
            backup_path = os.path.join(config.CKPT_DIR, backup_filename)
            torch.save(checkpoint_dict, backup_path)
            print(f"💾 Backup saved: {backup_filename}")

            # 시각화
            if val_hr_tensor is not None:
                model.eval()
                with torch.no_grad():
                    sr_tensor = model(val_lr_tensor, target_size=val_target_size)
                    sr_tensor = torch.clamp(sr_tensor, 0.0, 1.0)

                lr_np = val_lr_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
                sr_np = sr_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
                hr_np = val_hr_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()

                plt.figure(figsize=(15, 5))
                plt.subplot(1, 3, 1)
                plt.title(f"LR Input (Epoch {epoch+1})")
                plt.imshow(lr_np)
                plt.axis('off')

                plt.subplot(1, 3, 2)
                plt.title(f"LNO SR Output")
                plt.imshow(sr_np)
                plt.axis('off')

                plt.subplot(1, 3, 3)
                plt.title("Ground Truth")
                plt.imshow(hr_np)
                plt.axis('off')

                save_img_name = f"result_epoch_{epoch+1}.png"
                save_img_path = os.path.join(RESULT_PATH, save_img_name)
                plt.savefig(save_img_path, bbox_inches='tight')
                print(f"🖼️ Result image saved to: {save_img_path}")

                plt.show()
                model.train()

    print("🎉 Training Finished.")

# 실행 (Config.CKPT_NAME이 'epoch_00.pt'여도 내용 보고 제대로 이어서 합니다)
train_model(Config)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
import glob
import os

# 1. 설정 (Configuration)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SCALE_FACTOR = 4
# last_ckpt = "/content/drive/MyDrive/super_solutioner/LNO_base/checkpoints/epoch_190.pth"

# 2. 모델 준비
# 하이퍼파라미터는 학습 때와 똑같이 (Config 참조)
model = DeepLNO_SR(width=64, modes1=32, modes2=32).to(device)

if os.path.exists(last_ckpt):
    # [핵심 수정] 딕셔너리 구조 처리
    checkpoint = torch.load(last_ckpt, map_location=device)
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    print(f"✅ Loaded checkpoint: {last_ckpt}")
else:
    print(f"❌ File not found: {last_ckpt}")

model.eval()


# 3. 이미지 전처리 및 LR 생성 (Data Preprocessing)
# ==============================================================================
def load_and_preprocess(img_path, scale):
    hr_img = Image.open(img_path).convert('RGB')

    # 메모리 절약을 위해 너무 큰 이미지는 적당히 크롭 (예: 센터 512x512)
    # 전체 이미지를 하려면 이 부분을 주석 처리하세요.
    w, h = hr_img.size
    crop_size = 512
    if w > crop_size and h > crop_size:
        left = (w - crop_size) // 2
        top = (h - crop_size) // 2
        hr_img = hr_img.crop((left, top, left+crop_size, top+crop_size))

    # HR 크기 계산 (모델 입력은 짝수여야 FFT가 편안함)
    w_hr, h_hr = hr_img.size
    w_hr = w_hr - (w_hr % 2)
    h_hr = h_hr - (h_hr % 2)
    hr_img = hr_img.resize((w_hr, h_hr), Image.BICUBIC)

    # LR 이미지 생성 (Bicubic Downsample)
    w_lr = w_hr // scale
    h_lr = h_hr // scale
    lr_img = hr_img.resize((w_lr, h_lr), Image.BICUBIC)

    # Tensor 변환
    to_tensor = transforms.ToTensor()
    lr_tensor = to_tensor(lr_img).unsqueeze(0).to(device) # [1, 3, H_lr, W_lr]
    hr_tensor = to_tensor(hr_img).unsqueeze(0).to(device) # [1, 3, H_hr, W_hr]

    return lr_tensor, hr_tensor, (w_hr, h_hr)

if not os.path.exists(TEST_IMG_PATH):
    # 테스트 이미지가 없으면 다운로드 받은 것 중 첫 번째 사용
    valid_images = sorted(glob.glob(os.path.join(DATA_PATH, 'DIV2K_valid_HR', '*.png')))
    TEST_IMG_PATH = valid_images[0]

lr_img, hr_img, target_size = load_and_preprocess(TEST_IMG_PATH, SCALE_FACTOR)

# 4. 추론 실행 (Inference)
# ==============================================================================
print(f"Inference started... Input: {lr_img.shape}, Target: {target_size}")

with torch.no_grad():
    raw_output = model(lr_img, target_size=target_size)
    print(f"Raw Output Range: Min={raw_output.min()}, Max={raw_output.max()}")
    # 만약 Max가 1000을 넘는다면 1번, 2번 문제가 확실합니다.

    sr_img = torch.clamp(raw_output, 0.0, 1.0)

print(f"Inference finished. Output: {sr_img.shape}")

# 5. 결과 시각화 및 저장 (Visualization)
# ==============================================================================
def tensor_to_numpy(tensor):
    return tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()

lr_np = tensor_to_numpy(lr_img)
hr_np = tensor_to_numpy(hr_img)
sr_np = tensor_to_numpy(sr_img)

# Bicubic 결과와 비교를 위해 LR을 단순 확대
from cv2 import resize, INTER_CUBIC
import cv2
# OpenCV는 (W, H) 순서 주의
bicubic_np = cv2.resize(lr_np, (hr_np.shape[1], hr_np.shape[0]), interpolation=INTER_CUBIC)

plt.figure(figsize=(20, 10))

plt.subplot(1, 4, 1)
plt.title(f"Low Resolution (x1/{SCALE_FACTOR})")
plt.imshow(lr_np)
plt.axis('off')

plt.subplot(1, 4, 2)
plt.title("Bicubic Interpolation")
plt.imshow(bicubic_np)
plt.axis('off')

plt.subplot(1, 4, 3)
plt.title("LNO SR (Ours)")
plt.imshow(sr_np)
plt.axis('off')

plt.subplot(1, 4, 4)
plt.title("Ground Truth (HR)")
plt.imshow(hr_np)
plt.axis('off')

save_path = os.path.join(RESULT_PATH, 'inference_result.png')
plt.savefig(save_path, bbox_inches='tight')
print(f"Result saved to {save_path}")
plt.show()

# PSNR 계산 (간단한 성능 지표)
mse = np.mean((hr_np - sr_np) ** 2)
if mse == 0:
    psnr = 100
else:
    psnr = 20 * np.log10(1.0 / np.sqrt(mse))
print(f"PSNR: {psnr:.2f} dB")